# Generative Adversarial Networks (GANs): A Comprehensive Guide

---

## From Theory to Practice: Understanding, Implementing, and Applying GANs

This notebook provides an extensive, textbook-level treatment of **Generative Adversarial Networks (GANs)** — one of the most influential innovations in deep learning. We cover:

* **Historical context** and the intellectual lineage of GANs
* **Mathematical foundations** including game theory, probability, and optimization
* **Core architecture** and training dynamics
* **Major GAN variants** with detailed algorithms and working code
* **Industrial applications** with real-world case studies
* **Training challenges** and solutions

> *"The most interesting idea in the last 10 years in ML."* — Yann LeCun, on GANs (2016)

# Chapter 1: The History and Evolution of GANs

---

## 1.1 The Genesis (2014)

Generative Adversarial Networks were introduced by **Ian Goodfellow** and colleagues (Yoshua Bengio, Jean Pouget-Abadie, Mehdi Mirza, Bing Xu, David Warde-Farley, Sherjil Ozair, and Aaron Courville) in the landmark paper *"Generative Adversarial Nets"* (NeurIPS 2014).

The story goes that Goodfellow conceived the idea during a discussion at a bar in Montreal. His colleagues were working on generative models and struggling with the intractability of computing maximum likelihood estimates. Goodfellow proposed pitting two neural networks against each other — a **generator** that creates fake data and a **discriminator** that distinguishes real from fake.

## 1.2 Intellectual Lineage

GANs built upon several prior concepts:

| Year | Concept | Contribution |
| --- | --- | --- |
| 1943 | Game Theory (von Neumann) | Nash equilibrium, minimax strategies |
| 2006 | Deep Belief Networks (Hinton) | Layer-wise generative pre-training |
| 2013 | Variational Autoencoders (Kingma) | Latent space generative modeling |
| 2014 | **GANs (Goodfellow)** | Adversarial training framework |

## 1.3 Timeline of Key GAN Developments

| Year | Model | Key Innovation |
| --- | --- | --- |
| 2014 | Vanilla GAN | Original adversarial framework |
| 2014 | CGAN | Conditional generation with class labels |
| 2015 | DCGAN | Convolutional architectures for stable training |
| 2016 | InfoGAN | Disentangled representations via information maximization |
| 2016 | Pix2Pix | Paired image-to-image translation |
| 2017 | WGAN | Wasserstein distance for stable gradients |
| 2017 | CycleGAN | Unpaired image-to-image translation |
| 2017 | ProGAN | Progressive growing for high-resolution synthesis |
| 2018 | StyleGAN | Style-based architecture for controllable generation |
| 2019 | BigGAN | Large-scale class-conditional image generation |
| 2020 | StyleGAN2 | Improved quality, no progressive growing needed |
| 2021 | StyleGAN3 | Alias-free generation |

## 1.4 The GAN "Zoo"

By 2018, hundreds of GAN variants had been published. The "GAN Zoo" (maintained by Avinash Hindupur) cataloged over 500 named GAN architectures, reflecting the explosive research interest in adversarial training.

# Chapter 2: Mathematical Foundations

---

## 2.1 Probability and Generative Modeling

The fundamental goal of generative modeling is to learn the **true data distribution** $$p_{data}(x)$$ from a finite set of samples. A GAN achieves this by learning a mapping from a simple **prior distribution** $$p_z(z)$$ (typically Gaussian or Uniform) to the data space.

### Definitions:

* $$x$$ : A sample from the real data distribution $$p_{data}(x)$$
* $$z$$ : A latent vector sampled from prior $$p_z(z) = \mathcal{N}(0, I)$$
* $$G(z; \theta_g)$$ : Generator function parameterized by $$\theta_g$$
* $$D(x; \theta_d)$$ : Discriminator function parameterized by $$\theta_d$$
* $$p_g(x)$$ : The distribution implicitly defined by $$G(z)$$ when $$z \sim p_z$$

## 2.2 The Minimax Objective

The GAN training is formulated as a **two-player minimax game**:

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}(x)}[\log D(x)] + \mathbb{E}_{z \sim p_z(z)}[\log(1 - D(G(z)))]$$

### Intuition:
* **Discriminator** $$D$$ wants to **maximize** $$V(D,G)$$: assign high probability to real data ($$\log D(x) \to 0$$) and low probability to fake data ($$\log(1 - D(G(z))) \to 0$$)
* **Generator** $$G$$ wants to **minimize** $$V(D,G)$$: fool the discriminator so $$D(G(z)) \to 1$$

## 2.3 Optimal Discriminator

**Theorem (Goodfellow, 2014):** For a fixed generator $$G$$, the optimal discriminator is:

$$D^*_G(x) = \frac{p_{data}(x)}{p_{data}(x) + p_g(x)}$$

**Proof:**

The training criterion for $$D$$, given any generator $$G$$, is to maximize:

$$V(G, D) = \int_x p_{data}(x) \log(D(x)) \, dx + \int_z p_z(z) \log(1 - D(G(z))) \, dz$$

$$= \int_x \left[ p_{data}(x) \log(D(x)) + p_g(x) \log(1 - D(x)) \right] dx$$

For any $$(a, b) \in \mathbb{R}^2 \setminus \{0, 0\}$$, the function $$y \mapsto a \log(y) + b \log(1-y)$$ achieves its maximum at $$\frac{a}{a+b}$$. Thus:

$$D^*_G(x) = \frac{p_{data}(x)}{p_{data}(x) + p_g(x)} \quad \blacksquare$$

## 2.4 Global Optimum

**Theorem:** The global minimum of $$C(G) = \max_D V(G, D)$$ is achieved if and only if $$p_g = p_{data}$$, and the minimum value is $$-\log 4$$.

**Proof:** Substituting $$D^*_G$$ into $$V$$:

$$C(G) = \mathbb{E}_{x \sim p_{data}}\left[\log \frac{p_{data}(x)}{p_{data}(x) + p_g(x)}\right] + \mathbb{E}_{x \sim p_g}\left[\log \frac{p_g(x)}{p_{data}(x) + p_g(x)}\right]$$

$$= -\log 4 + KL\left(p_{data} \| \frac{p_{data} + p_g}{2}\right) + KL\left(p_g \| \frac{p_{data} + p_g}{2}\right)$$

$$= -\log 4 + 2 \cdot JSD(p_{data} \| p_g)$$

where $$JSD$$ is the **Jensen-Shannon Divergence**. Since $$JSD \geq 0$$ with equality iff $$p_{data} = p_g$$, we have $$C(G) \geq -\log 4$$ with equality at $$p_g = p_{data}$$. $$\blacksquare$$

## 2.5 Convergence Guarantee

Goodfellow et al. showed that if $$G$$ and $$D$$ have enough capacity (non-parametric setting), and the discriminator is allowed to reach its optimum at each step, then $$p_g$$ converges to $$p_{data}$$.

In practice, neural networks have finite capacity, and alternating gradient updates (rather than full optimization) are used — making convergence guarantees weaker but empirically effective.

# Chapter 3: GAN Architecture and Training Algorithm

---

## 3.1 The Two-Player Game

```
                    ┌─────────────────────┐
   Latent Space     │                     │
   z ~ N(0, I)  ──▶ │    GENERATOR G      │ ───▶ G(z) = Fake Data
                    │  (Neural Network)   │          │
                    └─────────────────────┘          │
                                                     │
                    ┌─────────────────────┐          │
   Real Data        │                     │          │
   x ~ p_data  ──▶ │  DISCRIMINATOR D    │ ◀────────┘
                    │  (Neural Network)   │
                    └─────────────────────┘
                              │
                              ▼
                    D(x) ∈ [0, 1]
                    (Real vs Fake probability)
```

## 3.2 The Generator

The generator $$G: \mathbb{R}^{d_z} \to \mathbb{R}^{d_x}$$ maps from a low-dimensional latent space to the high-dimensional data space:

* **Input**: Random noise vector $$z \in \mathbb{R}^{d_z}$$, typically $$d_z = 100$$
* **Output**: Synthetic data sample $$\hat{x} = G(z) \in \mathbb{R}^{d_x}$$
* **Goal**: Learn $$G$$ such that $$p_g \approx p_{data}$$

The generator is a differentiable function (neural network) that transforms simple noise into complex, structured outputs.

## 3.3 The Discriminator

The discriminator $$D: \mathbb{R}^{d_x} \to [0, 1]$$ is a binary classifier:

* **Input**: A data sample $$x$$ (either real or generated)
* **Output**: Probability that $$x$$ is real: $$D(x) = P(\text{real} | x)$$
* **Goal**: Correctly classify real vs. fake samples

## 3.4 Training Algorithm

**Algorithm 1: GAN Training (Goodfellow et al., 2014)**

---

**for** number of training iterations **do**:

&nbsp;&nbsp;&nbsp;&nbsp;**Step 1: Train Discriminator** ($$k$$ steps, typically $$k=1$$)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**for** $$k$$ steps **do**:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;• Sample minibatch of $$m$$ noise samples $$\{z^{(1)}, ..., z^{(m)}\}$$ from $$p_z(z)$$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;• Sample minibatch of $$m$$ real examples $$\{x^{(1)}, ..., x^{(m)}\}$$ from $$p_{data}(x)$$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;• Update discriminator by **ascending** its stochastic gradient:

$$\nabla_{\theta_d} \frac{1}{m} \sum_{i=1}^{m} \left[ \log D(x^{(i)}) + \log(1 - D(G(z^{(i)}))) \right]$$

&nbsp;&nbsp;&nbsp;&nbsp;**Step 2: Train Generator** (1 step)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;• Sample minibatch of $$m$$ noise samples $$\{z^{(1)}, ..., z^{(m)}\}$$ from $$p_z(z)$$

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;• Update generator by **descending** its stochastic gradient:

$$\nabla_{\theta_g} \frac{1}{m} \sum_{i=1}^{m} \log(1 - D(G(z^{(i)})))$$

**end for**

---

## 3.5 Non-Saturating Generator Loss

In practice, $$\log(1 - D(G(z)))$$ saturates early in training when $$D$$ easily rejects fake samples. Instead, we train $$G$$ to **maximize** $$\log D(G(z))$$:

$$\mathcal{L}_G = -\mathbb{E}_{z \sim p_z}[\log D(G(z))]$$

This provides stronger gradients early in training and is the standard implementation choice.

# Chapter 4: Implementation Setup

Before diving into implementations, we set up the necessary libraries and utilities that will be shared across all GAN variants.

In [0]:
%pip install torch torchvision matplotlib numpy --quiet

In [0]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Utility function to display generated images
def show_generated_images(images, num_images=16, title="Generated Images"):
    """Display a grid of generated images."""
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    fig.suptitle(title, fontsize=14)
    for i, ax in enumerate(axes.flat):
        if i < num_images and i < len(images):
            img = images[i].detach().cpu().numpy()
            if img.shape[0] == 1:  # Grayscale
                ax.imshow(img.squeeze(), cmap='gray')
            else:  # RGB
                ax.imshow(np.transpose(img, (1, 2, 0)))
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# Utility to track training progress
def plot_losses(g_losses, d_losses, title="Training Losses"):
    """Plot generator and discriminator losses over training."""
    plt.figure(figsize=(10, 5))
    plt.plot(g_losses, label='Generator Loss', alpha=0.7)
    plt.plot(d_losses, label='Discriminator Loss', alpha=0.7)
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

print("All utilities loaded successfully.")

In [0]:
# Load MNIST dataset - our primary dataset for demonstrations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # Normalize to [-1, 1]
])

train_dataset = torchvision.datasets.MNIST(
    root='/tmp/data', train=True, transform=transform, download=True
)

# DataLoader for training
batch_size = 64
dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

print(f"Dataset size: {len(train_dataset)} images")
print(f"Image shape: {train_dataset[0][0].shape}")
print(f"Number of batches: {len(dataloader)}")

# Visualize some real samples
real_batch = next(iter(dataloader))
show_generated_images(real_batch[0][:16], title="Real MNIST Samples")

# Chapter 5: Vanilla GAN — The Original Architecture

---

## 5.1 Architecture Overview

The original GAN (Goodfellow et al., 2014) uses fully-connected (dense) layers for both the generator and discriminator. While simple, it establishes all the fundamental principles.

## 5.2 Generator Architecture

$$G: \mathbb{R}^{100} \to \mathbb{R}^{784}$$

The generator maps a 100-dimensional noise vector to a 784-dimensional output (28×28 flattened image):

$$z \xrightarrow{\text{Linear}} h_1 \xrightarrow{\text{LeakyReLU}} h_2 \xrightarrow{\text{LeakyReLU}} h_3 \xrightarrow{\text{Tanh}} \hat{x}$$

## 5.3 Discriminator Architecture

$$D: \mathbb{R}^{784} \to [0, 1]$$

The discriminator maps a flattened image to a probability:

$$x \xrightarrow{\text{Linear}} h_1 \xrightarrow{\text{LeakyReLU}} h_2 \xrightarrow{\text{LeakyReLU}} h_3 \xrightarrow{\text{Sigmoid}} D(x)$$

## 5.4 Loss Functions

**Discriminator Loss:**

$$\mathcal{L}_D = -\frac{1}{m}\sum_{i=1}^{m}\left[\log D(x^{(i)}) + \log(1 - D(G(z^{(i)})))\right]$$

**Generator Loss (Non-Saturating):**

$$\mathcal{L}_G = -\frac{1}{m}\sum_{i=1}^{m}\log D(G(z^{(i)}))$$

## 5.5 Industrial Application: Synthetic Data Generation

**Use Case:** Financial institutions use Vanilla GANs to generate synthetic transaction data for fraud detection model training. Since fraud cases are rare (<0.1% of transactions), GANs can augment the minority class while preserving statistical properties.

**Example:** American Express uses GAN-generated synthetic data to test fraud detection systems without exposing real customer information, satisfying both model performance and privacy requirements.

In [0]:
# ============================================================
# VANILLA GAN - Complete Implementation
# ============================================================

class VanillaGenerator(nn.Module):
    """
    Generator network for Vanilla GAN.
    Maps latent vector z (dim=100) to image space (dim=784).
    """
    def __init__(self, latent_dim=100, img_dim=784):
        super(VanillaGenerator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(512),
            
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(1024),
            
            nn.Linear(1024, img_dim),
            nn.Tanh()  # Output in [-1, 1]
        )
    
    def forward(self, z):
        return self.model(z)


class VanillaDiscriminator(nn.Module):
    """
    Discriminator network for Vanilla GAN.
    Maps image (dim=784) to probability of being real.
    """
    def __init__(self, img_dim=784):
        super(VanillaDiscriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(img_dim, 1024),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(256, 1),
            nn.Sigmoid()  # Output probability
        )
    
    def forward(self, x):
        return self.model(x)


print("Vanilla GAN architecture defined.")
print(f"Generator parameters: {sum(p.numel() for p in VanillaGenerator().parameters()):,}")
print(f"Discriminator parameters: {sum(p.numel() for p in VanillaDiscriminator().parameters()):,}")

In [0]:
# ============================================================
# VANILLA GAN - Training
# ============================================================

def train_vanilla_gan(num_epochs=10, latent_dim=100, lr=0.0002):
    """
    Complete training loop for Vanilla GAN on MNIST.
    
    Training follows the alternating optimization:
    1. Update D to maximize log D(x) + log(1 - D(G(z)))
    2. Update G to maximize log D(G(z))
    """
    # Initialize models
    G = VanillaGenerator(latent_dim=latent_dim).to(device)
    D = VanillaDiscriminator().to(device)
    
    # Optimizers (Adam with beta1=0.5 as recommended)
    optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    
    # Binary Cross Entropy loss
    criterion = nn.BCELoss()
    
    # Track losses
    g_losses, d_losses = [], []
    
    # Fixed noise for visualization
    fixed_noise = torch.randn(16, latent_dim, device=device)
    
    print(f"Training Vanilla GAN for {num_epochs} epochs...")
    print("=" * 60)
    
    for epoch in range(num_epochs):
        for batch_idx, (real_images, _) in enumerate(dataloader):
            batch_size_curr = real_images.size(0)
            real_images = real_images.view(batch_size_curr, -1).to(device)  # Flatten
            
            # Labels
            real_labels = torch.ones(batch_size_curr, 1, device=device)
            fake_labels = torch.zeros(batch_size_curr, 1, device=device)
            
            # =====================
            # Train Discriminator
            # =====================
            optimizer_D.zero_grad()
            
            # Loss on real images
            outputs_real = D(real_images)
            d_loss_real = criterion(outputs_real, real_labels)
            
            # Loss on fake images
            z = torch.randn(batch_size_curr, latent_dim, device=device)
            fake_images = G(z)
            outputs_fake = D(fake_images.detach())  # detach to avoid backprop through G
            d_loss_fake = criterion(outputs_fake, fake_labels)
            
            # Total discriminator loss
            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            optimizer_D.step()
            
            # =====================
            # Train Generator
            # =====================
            optimizer_G.zero_grad()
            
            # Generate fake images and compute loss
            z = torch.randn(batch_size_curr, latent_dim, device=device)
            fake_images = G(z)
            outputs = D(fake_images)
            
            # Non-saturating loss: maximize log D(G(z))
            g_loss = criterion(outputs, real_labels)
            g_loss.backward()
            optimizer_G.step()
            
            # Track losses
            g_losses.append(g_loss.item())
            d_losses.append(d_loss.item())
        
        # Print progress
        print(f"Epoch [{epoch+1}/{num_epochs}] | "
              f"D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f} | "
              f"D(x): {outputs_real.mean().item():.3f} | D(G(z)): {outputs_fake.mean().item():.3f}")
    
    print("\nTraining Complete!")
    
    # Plot training curves
    plot_losses(g_losses, d_losses, "Vanilla GAN Training Losses")
    
    # Generate and display final samples
    G.eval()
    with torch.no_grad():
        generated = G(fixed_noise).view(-1, 1, 28, 28)
    show_generated_images(generated, title="Vanilla GAN - Generated MNIST Digits")
    
    return G, D

# Train the Vanilla GAN
vanilla_G, vanilla_D = train_vanilla_gan(num_epochs=10)

# Chapter 6: Deep Convolutional GAN (DCGAN)

---

## 6.1 Motivation

The Vanilla GAN uses fully-connected layers, which:
* Do not exploit spatial structure in images
* Require flattening, losing 2D relationships
* Scale poorly to higher resolutions

**DCGAN** (Radford, Metz, Chintala, 2015) introduced architectural guidelines for stable convolutional GANs.

## 6.2 Architecture Guidelines (The DCGAN Recipe)

1. **Replace pooling with strided convolutions** (discriminator) and fractional-strided convolutions/transposed convolutions (generator)
2. **Use BatchNorm** in both generator and discriminator (except output layer of G and input layer of D)
3. **Remove fully connected hidden layers** for deeper architectures
4. **Use ReLU** in generator (all layers except output which uses Tanh)
5. **Use LeakyReLU** in discriminator (all layers)

## 6.3 Generator Architecture (Fractional-Strided Convolutions)

The generator upsamples from a latent vector to a full image using transposed convolutions:

$$z \in \mathbb{R}^{100} \xrightarrow{\text{Reshape}} 4{\times}4{\times}1024 \xrightarrow{\text{ConvT}} 8{\times}8{\times}512 \xrightarrow{\text{ConvT}} 16{\times}16{\times}256 \xrightarrow{\text{ConvT}} 28{\times}28{\times}1$$

**Transposed Convolution Output Size:**

$$H_{out} = (H_{in} - 1) \times s - 2p + k + \text{output\_padding}$$

where $$s$$ = stride, $$p$$ = padding, $$k$$ = kernel size.

## 6.4 Discriminator Architecture (Strided Convolutions)

The discriminator downsamples the image to a single probability:

$$x \in \mathbb{R}^{28 \times 28 \times 1} \xrightarrow{\text{Conv}} 14{\times}14{\times}64 \xrightarrow{\text{Conv}} 7{\times}7{\times}128 \xrightarrow{\text{Flatten}} \xrightarrow{\text{Linear}} D(x) \in [0,1]$$

## 6.5 Weight Initialization

DCGAN uses specific weight initialization:

$$W \sim \mathcal{N}(0, 0.02)$$

All convolutional and batch normalization weights are initialized from a normal distribution with mean 0 and standard deviation 0.02.

## 6.6 Industrial Application: Image Super-Resolution & Content Creation

**Use Case:** Media companies (Netflix, Adobe) use DCGAN-style architectures for:
* **Content-Aware Fill:** Adobe Photoshop's generative fill uses convolutional GANs to inpaint missing regions
* **Asset Generation:** Game studios (EA, Ubisoft) generate texture variations for environments
* **Upscaling:** Anime and video streaming platforms upscale low-resolution content to 4K using DCGAN-based SRGAN

In [0]:
# ============================================================
# DCGAN - Deep Convolutional GAN Implementation
# ============================================================

def weights_init(m):
    """Custom weight initialization as per DCGAN paper."""
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class DCGenerator(nn.Module):
    """
    DCGAN Generator using transposed convolutions.
    Upsamples from latent vector (100,) to image (1, 28, 28).
    """
    def __init__(self, latent_dim=100, channels=1, features_g=64):
        super(DCGenerator, self).__init__()
        self.model = nn.Sequential(
            # Input: latent_dim x 1 x 1 -> features_g*4 x 4 x 4
            nn.ConvTranspose2d(latent_dim, features_g * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),
            
            # features_g*4 x 4 x 4 -> features_g*2 x 7 x 7
            nn.ConvTranspose2d(features_g * 4, features_g * 2, 4, 2, 2, bias=False),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),
            
            # features_g*2 x 7 x 7 -> features_g x 14 x 14
            nn.ConvTranspose2d(features_g * 2, features_g, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g),
            nn.ReLU(True),
            
            # features_g x 14 x 14 -> channels x 28 x 28
            nn.ConvTranspose2d(features_g, channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    
    def forward(self, z):
        return self.model(z)


class DCDiscriminator(nn.Module):
    """
    DCGAN Discriminator using strided convolutions.
    Downsamples from image (1, 28, 28) to probability scalar.
    """
    def __init__(self, channels=1, features_d=64):
        super(DCDiscriminator, self).__init__()
        self.model = nn.Sequential(
            # channels x 28 x 28 -> features_d x 14 x 14
            nn.Conv2d(channels, features_d, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            # features_d x 14 x 14 -> features_d*2 x 7 x 7
            nn.Conv2d(features_d, features_d * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace=True),
            
            # features_d*2 x 7 x 7 -> features_d*4 x 4 x 4
            nn.Conv2d(features_d * 2, features_d * 4, 3, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace=True),
            
            # features_d*4 x 4 x 4 -> 1 x 1 x 1
            nn.Conv2d(features_d * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x).view(-1, 1)


# Initialize models with DCGAN weight initialization
dc_G = DCGenerator().to(device)
dc_D = DCDiscriminator().to(device)
dc_G.apply(weights_init)
dc_D.apply(weights_init)

print("DCGAN Architecture:")
print(f"Generator parameters: {sum(p.numel() for p in dc_G.parameters()):,}")
print(f"Discriminator parameters: {sum(p.numel() for p in dc_D.parameters()):,}")
print(f"\nGenerator:\n{dc_G}")

In [0]:
# ============================================================
# DCGAN - Training Loop
# ============================================================

def train_dcgan(num_epochs=10, latent_dim=100, lr=0.0002):
    """
    Train DCGAN on MNIST with convolutional architecture.
    Key difference from Vanilla GAN: works on 2D image tensors directly.
    """
    G = DCGenerator(latent_dim=latent_dim).to(device)
    D = DCDiscriminator().to(device)
    G.apply(weights_init)
    D.apply(weights_init)
    
    optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()
    
    g_losses, d_losses = [], []
    fixed_noise = torch.randn(16, latent_dim, 1, 1, device=device)  # Note: 4D for conv
    
    print(f"Training DCGAN for {num_epochs} epochs...")
    print("=" * 60)
    
    for epoch in range(num_epochs):
        for batch_idx, (real_images, _) in enumerate(dataloader):
            batch_size_curr = real_images.size(0)
            real_images = real_images.to(device)  # Keep as 2D images (no flattening!)
            
            real_labels = torch.ones(batch_size_curr, 1, device=device)
            fake_labels = torch.zeros(batch_size_curr, 1, device=device)
            
            # === Train Discriminator ===
            optimizer_D.zero_grad()
            
            output_real = D(real_images)
            d_loss_real = criterion(output_real, real_labels)
            
            noise = torch.randn(batch_size_curr, latent_dim, 1, 1, device=device)
            fake_images = G(noise)
            output_fake = D(fake_images.detach())
            d_loss_fake = criterion(output_fake, fake_labels)
            
            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            optimizer_D.step()
            
            # === Train Generator ===
            optimizer_G.zero_grad()
            
            noise = torch.randn(batch_size_curr, latent_dim, 1, 1, device=device)
            fake_images = G(noise)
            output = D(fake_images)
            g_loss = criterion(output, real_labels)
            g_loss.backward()
            optimizer_G.step()
            
            g_losses.append(g_loss.item())
            d_losses.append(d_loss.item())
        
        print(f"Epoch [{epoch+1}/{num_epochs}] | "
              f"D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")
    
    print("\nTraining Complete!")
    plot_losses(g_losses, d_losses, "DCGAN Training Losses")
    
    G.eval()
    with torch.no_grad():
        generated = G(fixed_noise)
    show_generated_images(generated, title="DCGAN - Generated MNIST Digits")
    
    return G, D

# Train DCGAN
dc_G_trained, dc_D_trained = train_dcgan(num_epochs=10)

# Chapter 7: Conditional GAN (CGAN)

---

## 7.1 Motivation

Vanilla GANs and DCGANs generate images without any control over **what** is generated. We cannot specify "generate a digit 7" — we get random outputs from the learned distribution.

**Conditional GANs** (Mirza & Osindero, 2014) solve this by conditioning both the generator and discriminator on additional information $$y$$ (e.g., class labels, text, other images).

## 7.2 Mathematical Formulation

The minimax game becomes conditioned on $$y$$:

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}(x)}[\log D(x|y)] + \mathbb{E}_{z \sim p_z(z)}[\log(1 - D(G(z|y)|y))]$$

where:
* $$y$$ can be class labels, text embeddings, attributes, or any auxiliary information
* The generator becomes $$G(z, y)$$: maps noise **and** condition to data
* The discriminator becomes $$D(x, y)$$: judges if $$x$$ is real **given** condition $$y$$

## 7.3 Conditioning Mechanisms

### For Generator:
$$\text{input} = [z; y] = \text{concatenate}(z, \text{embed}(y))$$

The condition $$y$$ (one-hot encoded or embedded) is concatenated with the noise vector $$z$$.

### For Discriminator:
$$\text{input} = [x; y] = \text{concatenate}(x, \text{embed}(y))$$

The condition is concatenated with the input image (or its intermediate representation).

## 7.4 Industrial Application: Targeted Data Augmentation

**Use Case 1 - Medical Imaging:** Hospitals use CGANs to generate labeled medical images for rare conditions. For example, generating X-rays conditioned on specific pathologies (pneumonia, fracture type) to augment training data for diagnostic AI.

**Use Case 2 - E-Commerce:** Fashion retailers (Zalando, ASOS) use CGANs to generate product images conditioned on attributes like color, style, and size for virtual try-on applications.

**Use Case 3 - Automotive:** Self-driving car companies (Waymo, Tesla) generate synthetic driving scenarios conditioned on weather, time-of-day, and traffic density for simulation testing.

In [0]:
# ============================================================
# CONDITIONAL GAN (CGAN) - Implementation
# ============================================================

class ConditionalGenerator(nn.Module):
    """
    Conditional Generator: G(z, y) -> x
    Takes noise z and class label y to generate class-specific images.
    """
    def __init__(self, latent_dim=100, n_classes=10, img_dim=784, embed_dim=50):
        super(ConditionalGenerator, self).__init__()
        
        # Embedding layer for class labels
        self.label_embedding = nn.Embedding(n_classes, embed_dim)
        
        # Generator network (takes z + embedded label)
        self.model = nn.Sequential(
            nn.Linear(latent_dim + embed_dim, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(512),
            
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(1024),
            
            nn.Linear(1024, img_dim),
            nn.Tanh()
        )
    
    def forward(self, z, labels):
        # Embed labels and concatenate with noise
        label_embed = self.label_embedding(labels)
        gen_input = torch.cat([z, label_embed], dim=1)
        return self.model(gen_input)


class ConditionalDiscriminator(nn.Module):
    """
    Conditional Discriminator: D(x, y) -> [0, 1]
    Evaluates if image x is real given class label y.
    """
    def __init__(self, n_classes=10, img_dim=784, embed_dim=50):
        super(ConditionalDiscriminator, self).__init__()
        
        # Embedding layer for class labels
        self.label_embedding = nn.Embedding(n_classes, embed_dim)
        
        # Discriminator network (takes image + embedded label)
        self.model = nn.Sequential(
            nn.Linear(img_dim + embed_dim, 1024),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x, labels):
        # Embed labels and concatenate with image
        label_embed = self.label_embedding(labels)
        disc_input = torch.cat([x, label_embed], dim=1)
        return self.model(disc_input)


print("Conditional GAN architecture defined.")
print(f"Generator parameters: {sum(p.numel() for p in ConditionalGenerator().parameters()):,}")
print(f"Discriminator parameters: {sum(p.numel() for p in ConditionalDiscriminator().parameters()):,}")

In [0]:
# ============================================================
# CGAN - Training Loop with Conditional Generation
# ============================================================

def train_cgan(num_epochs=10, latent_dim=100, lr=0.0002):
    """
    Train Conditional GAN on MNIST.
    Key difference: both G and D receive class labels as additional input.
    """
    G = ConditionalGenerator(latent_dim=latent_dim).to(device)
    D = ConditionalDiscriminator().to(device)
    
    optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()
    
    g_losses, d_losses = [], []
    
    print(f"Training Conditional GAN for {num_epochs} epochs...")
    print("=" * 60)
    
    for epoch in range(num_epochs):
        for batch_idx, (real_images, labels) in enumerate(dataloader):
            batch_size_curr = real_images.size(0)
            real_images = real_images.view(batch_size_curr, -1).to(device)
            labels = labels.to(device)
            
            real_labels_target = torch.ones(batch_size_curr, 1, device=device)
            fake_labels_target = torch.zeros(batch_size_curr, 1, device=device)
            
            # === Train Discriminator ===
            optimizer_D.zero_grad()
            
            # Real images with correct labels
            output_real = D(real_images, labels)
            d_loss_real = criterion(output_real, real_labels_target)
            
            # Fake images with the same labels (G should produce matching images)
            z = torch.randn(batch_size_curr, latent_dim, device=device)
            fake_images = G(z, labels)
            output_fake = D(fake_images.detach(), labels)
            d_loss_fake = criterion(output_fake, fake_labels_target)
            
            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            optimizer_D.step()
            
            # === Train Generator ===
            optimizer_G.zero_grad()
            
            z = torch.randn(batch_size_curr, latent_dim, device=device)
            fake_images = G(z, labels)
            output = D(fake_images, labels)
            g_loss = criterion(output, real_labels_target)
            g_loss.backward()
            optimizer_G.step()
            
            g_losses.append(g_loss.item())
            d_losses.append(d_loss.item())
        
        print(f"Epoch [{epoch+1}/{num_epochs}] | "
              f"D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")
    
    print("\nTraining Complete!")
    plot_losses(g_losses, d_losses, "Conditional GAN Training Losses")
    
    # Generate specific digits (the power of conditioning!)
    G.eval()
    print("\nGenerating specific digits (0-9):")
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    fig.suptitle("CGAN: Generating Specific Digits (0-9)", fontsize=14)
    
    with torch.no_grad():
        for digit in range(10):
            z = torch.randn(1, latent_dim, device=device)
            label = torch.tensor([digit], device=device)
            generated_img = G(z, label).view(28, 28).cpu().numpy()
            
            ax = axes[digit // 5, digit % 5]
            ax.imshow(generated_img, cmap='gray')
            ax.set_title(f"Digit: {digit}")
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return G, D

# Train CGAN
cgan_G, cgan_D = train_cgan(num_epochs=10)

# Chapter 8: Wasserstein GAN (WGAN)

---

## 8.1 The Problem with Standard GAN Loss

Training standard GANs suffers from several issues:

1. **Vanishing Gradients:** When the discriminator is too good, $$D(G(z)) \approx 0$$, and $$\log(1 - D(G(z))) \to 0$$, providing no gradient signal to $$G$$
2. **Mode Collapse:** The generator may learn to produce only a few modes of the data distribution
3. **Training Instability:** The minimax game often oscillates without converging

The root cause: the **Jensen-Shannon Divergence** (used implicitly by standard GANs) is not continuous when the supports of $$p_{data}$$ and $$p_g$$ don't overlap.

## 8.2 Wasserstein Distance (Earth Mover's Distance)

WGAN (Arjovsky, Chintala, Bottou, 2017) replaces JSD with the **Wasserstein-1 distance**:

$$W(p_{data}, p_g) = \inf_{\gamma \in \Pi(p_{data}, p_g)} \mathbb{E}_{(x,y) \sim \gamma}[\|x - y\|]$$

**Intuition:** The minimum cost of transporting mass from distribution $$p_g$$ to $$p_{data}$$, where cost is proportional to distance moved.

### Why Wasserstein is Better:

| Property | JSD | Wasserstein |
| --- | --- | --- |
| When supports don't overlap | Constant ($$\log 2$$) | Smooth, meaningful gradient |
| Gradient quality | Vanishes | Always informative |
| Correlation with sample quality | Weak | Strong |

## 8.3 Kantorovich-Rubinstein Duality

Direct computation of $$W$$ is intractable. Using the **Kantorovich-Rubinstein duality**:

$$W(p_{data}, p_g) = \sup_{\|f\|_L \leq 1} \mathbb{E}_{x \sim p_{data}}[f(x)] - \mathbb{E}_{x \sim p_g}[f(x)]$$

where the supremum is over all **1-Lipschitz functions** $$f$$.

A function $$f$$ is $$K$$-Lipschitz if:

$$|f(x_1) - f(x_2)| \leq K \cdot \|x_1 - x_2\| \quad \forall x_1, x_2$$

## 8.4 The WGAN Objective

Replace the discriminator with a **critic** $$f_w$$ (no sigmoid, unbounded output):

$$\max_w \mathbb{E}_{x \sim p_{data}}[f_w(x)] - \mathbb{E}_{z \sim p_z}[f_w(G(z))]$$

$$\min_\theta \quad -\mathbb{E}_{z \sim p_z}[f_w(G_\theta(z))]$$

### Enforcing Lipschitz Constraint:

**WGAN (original):** Weight clipping — clamp all weights $$w$$ to $$[-c, c]$$ after each update

**WGAN-GP (improved):** Gradient penalty — add a penalty term:

$$\mathcal{L}_{GP} = \lambda \mathbb{E}_{\hat{x} \sim p_{\hat{x}}}\left[(\|\nabla_{\hat{x}} f_w(\hat{x})\|_2 - 1)^2\right]$$

where $$\hat{x} = \epsilon x + (1 - \epsilon) G(z)$$ is interpolated between real and fake samples, $$\epsilon \sim U(0, 1)$$.

## 8.5 Key WGAN Differences from Standard GAN

| Aspect | Standard GAN | WGAN |
| --- | --- | --- |
| Output layer | Sigmoid (probability) | Linear (score) |
| Loss function | Binary Cross-Entropy | Wasserstein distance |
| Name | Discriminator | Critic |
| Critic updates per G update | 1 | 5 (typically) |
| Optimizer | Adam | RMSProp (original) / Adam (WGAN-GP) |
| Lipschitz constraint | None | Weight clipping / Gradient penalty |

## 8.6 Industrial Application: Drug Discovery & Molecular Generation

**Use Case:** Pharmaceutical companies (Insilico Medicine, Recursion Pharma) use WGAN variants for molecular generation. The Wasserstein distance provides meaningful gradients even when the generated molecule distribution doesn't overlap with known drug-like molecules — critical for exploring novel chemical spaces.

**Example:** Insilico Medicine used WGAN-based architectures in their AI platform to discover a novel drug candidate for idiopathic pulmonary fibrosis (IPF) that entered Phase I clinical trials in 2021 — one of the first AI-designed drugs in clinical testing.

In [0]:
# ============================================================
# WASSERSTEIN GAN with GRADIENT PENALTY (WGAN-GP)
# ============================================================

class WGANCritic(nn.Module):
    """
    WGAN Critic (NOT discriminator - no sigmoid!).
    Outputs an unbounded real-valued score.
    """
    def __init__(self, img_dim=784):
        super(WGANCritic, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(img_dim, 512),
            nn.LeakyReLU(0.2),
            
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            
            nn.Linear(128, 1)  # NO sigmoid! Unbounded output.
        )
    
    def forward(self, x):
        return self.model(x)


class WGANGenerator(nn.Module):
    """
    Generator for WGAN (same architecture as Vanilla, different training).
    """
    def __init__(self, latent_dim=100, img_dim=784):
        super(WGANGenerator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(512),
            
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(1024),
            
            nn.Linear(1024, img_dim),
            nn.Tanh()
        )
    
    def forward(self, z):
        return self.model(z)


def compute_gradient_penalty(critic, real_samples, fake_samples, device):
    """
    Compute gradient penalty for WGAN-GP.
    
    Enforces the Lipschitz constraint by penalizing the gradient norm
    of the critic's output w.r.t. interpolated samples.
    
    The penalty encourages ||∇_x̂ f(x̂)||₂ ≈ 1
    """
    # Random interpolation coefficient
    epsilon = torch.rand(real_samples.size(0), 1, device=device)
    
    # Interpolated samples: x̂ = ε*real + (1-ε)*fake
    interpolated = (epsilon * real_samples + (1 - epsilon) * fake_samples).requires_grad_(True)
    
    # Critic output on interpolated samples
    critic_interpolated = critic(interpolated)
    
    # Compute gradients
    gradients = torch.autograd.grad(
        outputs=critic_interpolated,
        inputs=interpolated,
        grad_outputs=torch.ones_like(critic_interpolated),
        create_graph=True,
        retain_graph=True
    )[0]
    
    # Gradient penalty: (||∇||₂ - 1)²
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    
    return gradient_penalty


print("WGAN-GP components defined.")
print(f"Critic parameters: {sum(p.numel() for p in WGANCritic().parameters()):,}")
print(f"Generator parameters: {sum(p.numel() for p in WGANGenerator().parameters()):,}")

In [0]:
# ============================================================
# WGAN-GP - Training Loop
# ============================================================

def train_wgan_gp(num_epochs=10, latent_dim=100, lr=0.0001, 
                  n_critic=5, lambda_gp=10):
    """
    Train WGAN with Gradient Penalty.
    
    Key differences from standard GAN training:
    1. Critic (not discriminator) trained n_critic times per G update
    2. Wasserstein loss (no log, no BCE)
    3. Gradient penalty enforces Lipschitz constraint
    4. No sigmoid in critic output
    """
    G = WGANGenerator(latent_dim=latent_dim).to(device)
    C = WGANCritic().to(device)  # Critic, not discriminator!
    
    # Adam with specific betas for WGAN-GP
    optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(0.0, 0.9))
    optimizer_C = optim.Adam(C.parameters(), lr=lr, betas=(0.0, 0.9))
    
    g_losses, c_losses, w_distances = [], [], []
    fixed_noise = torch.randn(16, latent_dim, device=device)
    
    print(f"Training WGAN-GP for {num_epochs} epochs...")
    print(f"Critic updates per generator update: {n_critic}")
    print(f"Gradient penalty coefficient (lambda): {lambda_gp}")
    print("=" * 60)
    
    for epoch in range(num_epochs):
        for batch_idx, (real_images, _) in enumerate(dataloader):
            batch_size_curr = real_images.size(0)
            real_images = real_images.view(batch_size_curr, -1).to(device)
            
            # =====================
            # Train Critic (n_critic steps)
            # =====================
            for _ in range(n_critic):
                optimizer_C.zero_grad()
                
                # Critic scores on real data
                critic_real = C(real_images)
                
                # Critic scores on fake data
                z = torch.randn(batch_size_curr, latent_dim, device=device)
                fake_images = G(z).detach()
                critic_fake = C(fake_images)
                
                # Gradient penalty
                gp = compute_gradient_penalty(C, real_images, fake_images, device)
                
                # WGAN-GP Critic Loss:
                # maximize E[C(real)] - E[C(fake)] - lambda * GP
                # (we minimize the negative)
                c_loss = -(critic_real.mean() - critic_fake.mean()) + lambda_gp * gp
                c_loss.backward()
                optimizer_C.step()
            
            # =====================
            # Train Generator (1 step)
            # =====================
            optimizer_G.zero_grad()
            
            z = torch.randn(batch_size_curr, latent_dim, device=device)
            fake_images = G(z)
            critic_fake = C(fake_images)
            
            # Generator loss: maximize E[C(G(z))] (minimize negative)
            g_loss = -critic_fake.mean()
            g_loss.backward()
            optimizer_G.step()
            
            # Track metrics
            g_losses.append(g_loss.item())
            c_losses.append(c_loss.item())
            # Wasserstein distance estimate
            w_dist = critic_real.mean().item() - critic_fake.mean().item()
            w_distances.append(w_dist)
        
        print(f"Epoch [{epoch+1}/{num_epochs}] | "
              f"C Loss: {c_loss.item():.4f} | G Loss: {g_loss.item():.4f} | "
              f"W-Distance: {w_dist:.4f}")
    
    print("\nTraining Complete!")
    
    # Plot Wasserstein distance (correlates with sample quality!)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(g_losses, label='Generator Loss', alpha=0.7)
    ax1.plot(c_losses, label='Critic Loss', alpha=0.7)
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Loss')
    ax1.set_title('WGAN-GP Training Losses')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(w_distances, color='green', alpha=0.7)
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Wasserstein Distance')
    ax2.set_title('Wasserstein Distance (correlates with quality)')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Generate samples
    G.eval()
    with torch.no_grad():
        generated = G(fixed_noise).view(-1, 1, 28, 28)
    show_generated_images(generated, title="WGAN-GP - Generated MNIST Digits")
    
    return G, C

# Train WGAN-GP
wgan_G, wgan_C = train_wgan_gp(num_epochs=10)

# Chapter 9: StyleGAN — Style-Based Generator Architecture

---

## 9.1 Overview

**StyleGAN** (Karras et al., NVIDIA, 2018) revolutionized high-resolution image synthesis by borrowing ideas from **style transfer**. It introduced unprecedented control over the generation process through a novel generator architecture.

## 9.2 Key Innovations

### 9.2.1 Mapping Network

Instead of feeding $$z$$ directly to the generator, StyleGAN uses a **mapping network** $$f: \mathcal{Z} \to \mathcal{W}$$:

$$w = f(z) \quad \text{where } f \text{ is an 8-layer MLP}$$

The intermediate latent space $$\mathcal{W}$$ is less entangled than $$\mathcal{Z}$$, meaning individual dimensions in $$w$$ correspond more directly to individual attributes.

### 9.2.2 Adaptive Instance Normalization (AdaIN)

The style vector $$w$$ modulates features via AdaIN at each layer:

$$\text{AdaIN}(x_i, y) = y_{s,i} \frac{x_i - \mu(x_i)}{\sigma(x_i)} + y_{b,i}$$

where:
* $$x_i$$ is the feature map at layer $$i$$
* $$y_{s,i}$$ and $$y_{b,i}$$ are scale and bias derived from $$w$$ via learned affine transformations
* $$\mu(x_i)$$ and $$\sigma(x_i)$$ are per-channel mean and standard deviation

### 9.2.3 Style Mixing (Mixing Regularization)

During training, two latent codes $$z_1, z_2$$ are mapped to $$w_1, w_2$$. Different layers receive different styles:

* **Coarse styles** (4×4 – 8×8): pose, face shape, glasses
* **Middle styles** (16×16 – 32×32): facial features, hair style
* **Fine styles** (64×64 – 1024×1024): color, micro-structure

### 9.2.4 Noise Injection

Per-pixel Gaussian noise is added after each convolution:

$$x' = x + B \cdot n$$

where $$n \sim \mathcal{N}(0, I)$$ and $$B$$ is a learned per-channel scaling factor. This controls stochastic variation (hair placement, freckles, pores).

## 9.3 Progressive Growing (from ProGAN)

StyleGAN builds on **Progressive GAN** (Karras et al., 2017) which trains starting from low resolution and progressively adds layers:

$$4{\times}4 \to 8{\times}8 \to 16{\times}16 \to \cdots \to 1024{\times}1024$$

This stabilizes training by learning coarse structure first, then refining details.

## 9.4 StyleGAN2 Improvements

1. **Weight demodulation** replaces AdaIN (removes blob artifacts)
2. **Path length regularization** encourages smooth mappings
3. **No progressive growing** — uses skip connections instead
4. **Lazy regularization** — apply regularization every 16 minibatches

## 9.5 Industrial Application: Photorealistic Face Generation & Virtual Avatars

**Use Case 1 - Privacy:** Companies like Generated Photos sell AI-generated faces for marketing materials, avoiding privacy issues with real photographs.

**Use Case 2 - Gaming & Metaverse:** Epic Games (MetaHuman), NVIDIA (Omniverse) use StyleGAN-derived architectures to create photorealistic virtual characters.

**Use Case 3 - Fashion:** Zalando and other fashion retailers use StyleGAN to generate model images wearing different outfits, reducing the need for expensive photoshoots.

**Use Case 4 - Real Estate:** Architectural visualization firms use StyleGAN variants to generate photorealistic interior designs from floor plans.

In [0]:
# ============================================================
# STYLEGAN - Simplified Implementation (Key Concepts)
# ============================================================
# Full StyleGAN requires significant compute (TPU/multi-GPU).
# Here we implement the KEY architectural innovations as modules.

class MappingNetwork(nn.Module):
    """
    StyleGAN Mapping Network: z -> w
    8-layer MLP that maps from Z space to W space.
    W space is more disentangled, allowing finer control.
    """
    def __init__(self, z_dim=512, w_dim=512, num_layers=8):
        super(MappingNetwork, self).__init__()
        layers = []
        for i in range(num_layers):
            in_dim = z_dim if i == 0 else w_dim
            layers.extend([
                nn.Linear(in_dim, w_dim),
                nn.LeakyReLU(0.2)
            ])
        self.mapping = nn.Sequential(*layers)
    
    def forward(self, z):
        return self.mapping(z)


class AdaIN(nn.Module):
    """
    Adaptive Instance Normalization.
    Modulates feature statistics using the style vector w.
    
    AdaIN(x, y) = y_s * (x - mu(x)) / sigma(x) + y_b
    """
    def __init__(self, channels, w_dim=512):
        super(AdaIN, self).__init__()
        self.instance_norm = nn.InstanceNorm2d(channels)
        # Affine transform: w -> (scale, bias) for each channel
        self.style_scale = nn.Linear(w_dim, channels)
        self.style_bias = nn.Linear(w_dim, channels)
    
    def forward(self, x, w):
        # Normalize
        x = self.instance_norm(x)
        # Compute style parameters
        scale = self.style_scale(w).unsqueeze(2).unsqueeze(3)  # (B, C, 1, 1)
        bias = self.style_bias(w).unsqueeze(2).unsqueeze(3)
        # Apply style
        return scale * x + bias


class NoiseInjection(nn.Module):
    """
    Per-pixel noise injection for stochastic variation.
    Controls random details like hair strand placement, skin pores.
    """
    def __init__(self, channels):
        super(NoiseInjection, self).__init__()
        self.weight = nn.Parameter(torch.zeros(1, channels, 1, 1))
    
    def forward(self, x):
        noise = torch.randn(x.size(0), 1, x.size(2), x.size(3), device=x.device)
        return x + self.weight * noise


class StyleBlock(nn.Module):
    """
    A single synthesis block in StyleGAN:
    Conv -> Noise -> AdaIN -> Activation
    """
    def __init__(self, in_channels, out_channels, w_dim=512):
        super(StyleBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, 3, 1, 1)
        self.noise = NoiseInjection(out_channels)
        self.adain = AdaIN(out_channels, w_dim)
        self.activation = nn.LeakyReLU(0.2)
    
    def forward(self, x, w):
        x = self.conv(x)
        x = self.noise(x)
        x = self.adain(x, w)
        x = self.activation(x)
        return x


class SimplifiedStyleGenerator(nn.Module):
    """
    Simplified StyleGAN Generator for MNIST (28x28).
    Demonstrates the key concepts:
    1. Mapping network (z -> w)
    2. Constant learned input
    3. Style modulation via AdaIN at each layer
    4. Noise injection for stochastic details
    """
    def __init__(self, z_dim=100, w_dim=256):
        super(SimplifiedStyleGenerator, self).__init__()
        
        # Mapping network: Z -> W
        self.mapping = nn.Sequential(
            nn.Linear(z_dim, w_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(w_dim, w_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(w_dim, w_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(w_dim, w_dim),
            nn.LeakyReLU(0.2),
        )
        
        # Learned constant input (4x4)
        self.constant_input = nn.Parameter(torch.randn(1, 256, 4, 4))
        
        # Style blocks with progressive upsampling
        self.style_block1 = StyleBlock(256, 128, w_dim)  # 4x4 -> 4x4
        self.up1 = nn.Upsample(scale_factor=2)           # 4x4 -> 8x8
        self.style_block2 = StyleBlock(128, 64, w_dim)   # 8x8 -> 8x8
        self.up2 = nn.Upsample(size=(14, 14))            # 8x8 -> 14x14
        self.style_block3 = StyleBlock(64, 32, w_dim)    # 14x14 -> 14x14
        self.up3 = nn.Upsample(scale_factor=2)           # 14x14 -> 28x28
        
        # Final output (to image)
        self.to_rgb = nn.Sequential(
            nn.Conv2d(32, 1, 1),
            nn.Tanh()
        )
    
    def forward(self, z):
        # Map z to style space w
        w = self.mapping(z)
        
        # Start from constant input
        batch_size = z.size(0)
        x = self.constant_input.expand(batch_size, -1, -1, -1)
        
        # Progressive synthesis with style modulation
        x = self.style_block1(x, w)
        x = self.up1(x)
        x = self.style_block2(x, w)
        x = self.up2(x)
        x = self.style_block3(x, w)
        x = self.up3(x)
        
        # Convert to image
        x = self.to_rgb(x)
        return x


# Demonstrate the architecture
style_gen = SimplifiedStyleGenerator().to(device)
z_test = torch.randn(4, 100, device=device)
with torch.no_grad():
    output = style_gen(z_test)
print(f"StyleGAN-like Generator:")
print(f"  Input: z shape = {z_test.shape}")
print(f"  Output: image shape = {output.shape}")
print(f"  Total parameters: {sum(p.numel() for p in style_gen.parameters()):,}")
print(f"\nKey Components:")
print(f"  - Mapping Network: z (100-dim) -> w (256-dim)")
print(f"  - Constant Input: learned 4x4 feature maps")
print(f"  - Style Blocks: Conv + Noise + AdaIN + ReLU")
print(f"  - Progressive Upsampling: 4x4 -> 8x8 -> 14x14 -> 28x28")

# Chapter 10: CycleGAN — Unpaired Image-to-Image Translation

---

## 10.1 The Problem: Unpaired Translation

**Pix2Pix** (Isola et al., 2016) requires **paired** training data — for each input image, a corresponding output must exist (e.g., edges ↔ photos). Obtaining paired data is expensive or impossible for many domains.

**CycleGAN** (Zhu et al., 2017) enables translation between domains $$X$$ and $$Y$$ using only **unpaired** collections from each domain.

Examples: horses ↔ zebras, summer ↔ winter, photos ↔ paintings, day ↔ night

## 10.2 Architecture: Two Generator-Discriminator Pairs

CycleGAN uses two generators and two discriminators:

* $$G: X \to Y$$ (translates from domain X to domain Y)
* $$F: Y \to X$$ (translates from domain Y to domain X)
* $$D_X$$: discriminator for domain X
* $$D_Y$$: discriminator for domain Y

## 10.3 The Cycle Consistency Loss

The key insight: if we translate from $$X \to Y$$ and back $$Y \to X$$, we should recover the original:

**Forward Cycle:**
$$x \to G(x) \to F(G(x)) \approx x$$

**Backward Cycle:**
$$y \to F(y) \to G(F(y)) \approx y$$

**Cycle Consistency Loss:**

$$\mathcal{L}_{cyc}(G, F) = \mathbb{E}_{x \sim p_{data}(x)}[\|F(G(x)) - x\|_1] + \mathbb{E}_{y \sim p_{data}(y)}[\|G(F(y)) - y\|_1]$$

## 10.4 Full Objective

$$\mathcal{L}(G, F, D_X, D_Y) = \mathcal{L}_{GAN}(G, D_Y, X, Y) + \mathcal{L}_{GAN}(F, D_X, Y, X) + \lambda \mathcal{L}_{cyc}(G, F)$$

where:

$$\mathcal{L}_{GAN}(G, D_Y, X, Y) = \mathbb{E}_{y \sim p_{data}(y)}[\log D_Y(y)] + \mathbb{E}_{x \sim p_{data}(x)}[\log(1 - D_Y(G(x)))]$$

The full optimization:

$$G^*, F^* = \arg\min_{G,F} \max_{D_X, D_Y} \mathcal{L}(G, F, D_X, D_Y)$$

## 10.5 Identity Loss (Optional Regularization)

To preserve color composition:

$$\mathcal{L}_{identity} = \mathbb{E}_{y \sim p_{data}(y)}[\|G(y) - y\|_1] + \mathbb{E}_{x \sim p_{data}(x)}[\|F(x) - x\|_1]$$

If a sample from domain $$Y$$ is fed to $$G$$ (which maps $$X \to Y$$), it should remain unchanged.

## 10.6 Industrial Application: Domain Adaptation & Style Transfer

**Use Case 1 - Autonomous Driving:** Waymo and Cruise use CycleGAN to translate simulation renderings into photorealistic images, enabling sim-to-real transfer for training perception models without expensive real-world data collection.

**Use Case 2 - Medical Imaging:** Translating between MRI modalities (T1 ↔ T2 weighted) or CT ↔ MRI for clinics that have limited scanner access. Siemens Healthineers explored CycleGAN for cross-modality synthesis.

**Use Case 3 - Satellite Imagery:** Mapping daytime satellite images to nighttime (and vice versa) for defense and urban planning applications.

**Use Case 4 - Art & Film:** Converting footage between artistic styles (live-action ↔ anime), used in production pipelines at studios like Studio Ghibli and Netflix Anime.

In [0]:
# ============================================================
# CYCLEGAN - Core Architecture Components
# ============================================================

class ResidualBlock(nn.Module):
    """
    Residual block used in CycleGAN generators.
    Preserves spatial dimensions while learning residual transformations.
    """
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels)
        )
    
    def forward(self, x):
        return x + self.block(x)  # Skip connection


class CycleGenerator(nn.Module):
    """
    CycleGAN Generator: Encoder -> Transformer -> Decoder
    Uses residual blocks for the transformation.
    Architecture: c7s1-64, d128, d256, R256x6, u128, u64, c7s1-3
    """
    def __init__(self, input_channels=1, output_channels=1, n_residual=6):
        super(CycleGenerator, self).__init__()
        
        # Initial convolution
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_channels, 64, 7),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True)
        ]
        
        # Downsampling (Encoder)
        in_features = 64
        out_features = in_features * 2
        for _ in range(2):
            model += [
                nn.Conv2d(in_features, out_features, 3, stride=2, padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features * 2
        
        # Residual blocks (Transformer)
        for _ in range(n_residual):
            model += [ResidualBlock(in_features)]
        
        # Upsampling (Decoder)
        out_features = in_features // 2
        for _ in range(2):
            model += [
                nn.ConvTranspose2d(in_features, out_features, 3, stride=2, 
                                   padding=1, output_padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features // 2
        
        # Output layer
        model += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(64, output_channels, 7),
            nn.Tanh()
        ]
        
        self.model = nn.Sequential(*model)
    
    def forward(self, x):
        return self.model(x)


class CycleDiscriminator(nn.Module):
    """
    PatchGAN Discriminator (70x70 receptive field).
    Instead of classifying the whole image, classifies overlapping patches.
    Output is a feature map where each value represents real/fake for a patch.
    """
    def __init__(self, input_channels=1):
        super(CycleDiscriminator, self).__init__()
        
        self.model = nn.Sequential(
            nn.Conv2d(input_channels, 64, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(256, 1, 4, padding=1)  # PatchGAN output
        )
    
    def forward(self, x):
        return self.model(x)


# Demonstrate architecture
cycle_G = CycleGenerator(input_channels=1, output_channels=1, n_residual=4)
cycle_D = CycleDiscriminator(input_channels=1)

test_input = torch.randn(1, 1, 28, 28)
with torch.no_grad():
    gen_output = cycle_G(test_input)
    disc_output = cycle_D(test_input)

print("CycleGAN Architecture:")
print(f"  Generator: {test_input.shape} -> {gen_output.shape}")
print(f"  Discriminator (PatchGAN): {test_input.shape} -> {disc_output.shape}")
print(f"  Generator params: {sum(p.numel() for p in cycle_G.parameters()):,}")
print(f"  Discriminator params: {sum(p.numel() for p in cycle_D.parameters()):,}")
print(f"\n  Note: PatchGAN output is a spatial map, not a single value.")
print(f"  Each spatial location classifies a local patch as real/fake.")

# Chapter 11: Other Important GAN Variants

---

## 11.1 InfoGAN — Information Maximizing GAN (Chen et al., 2016)

### Motivation
InfoGAN learns **disentangled representations** in a completely unsupervised manner by maximizing the mutual information between a subset of latent variables and the generated output.

### Formulation

The latent input is split into:
* $$z$$: Incompressible noise (standard GAN noise)
* $$c$$: Latent code representing meaningful attributes

The objective adds an **information regularization** term:

$$\min_G \max_D V_I(D, G) = V(D, G) - \lambda I(c; G(z, c))$$

where $$I(c; G(z, c))$$ is the **mutual information** between the code $$c$$ and the generated output.

Since $$I(c; G(z, c))$$ is hard to compute directly, InfoGAN uses a **variational lower bound**:

$$I(c; G(z, c)) \geq \mathbb{E}_{c \sim p(c), x \sim G(z,c)}[\log Q(c|x)] + H(c) = L_I(G, Q)$$

where $$Q(c|x)$$ is an auxiliary network that approximates $$P(c|x)$$.

### Industrial Application
InfoGAN is used in **product design** — generating furniture/clothing variations where each latent dimension controls a specific attribute (e.g., width, color, style) without labels.

---

## 11.2 Pix2Pix — Paired Image-to-Image Translation (Isola et al., 2016)

### Formulation

Given paired training data $$(x, y)$$ where $$x$$ is the input and $$y$$ is the target:

$$\mathcal{L}_{cGAN}(G, D) = \mathbb{E}_{x,y}[\log D(x, y)] + \mathbb{E}_{x,z}[\log(1 - D(x, G(x, z)))]$$

Plus an L1 reconstruction loss:

$$\mathcal{L}_{L1}(G) = \mathbb{E}_{x,y,z}[\|y - G(x, z)\|_1]$$

Full objective:

$$G^* = \arg\min_G \max_D \mathcal{L}_{cGAN}(G, D) + \lambda \mathcal{L}_{L1}(G)$$

### Key Innovation: U-Net Generator + PatchGAN Discriminator

* **U-Net**: Encoder-decoder with skip connections, preserving fine details
* **PatchGAN**: Discriminates at the patch level (local texture quality)

### Industrial Application
* **Architecture:** Converting sketches to photorealistic building renderings
* **Satellite Imaging:** Map tiles to aerial photographs (Google Maps)
* **Medical:** Converting segmentation masks to realistic tissue images

---

## 11.3 Progressive GAN (ProGAN) — Karras et al., 2017

### Key Idea

Train both G and D starting from very low resolution, progressively adding layers:

$$\text{Start: } 4{\times}4 \xrightarrow{\text{add layers}} 8{\times}8 \xrightarrow{} 16{\times}16 \xrightarrow{} \cdots \xrightarrow{} 1024{\times}1024$$

### Smooth Fade-In

New layers are blended in smoothly using parameter $$\alpha$$ that goes from 0 to 1:

$$\text{output} = (1 - \alpha) \cdot \text{upsampled\_old} + \alpha \cdot \text{new\_layer\_output}$$

### Why It Works
* Low-res training is fast and captures global structure
* Adding layers incrementally is more stable than training high-res from scratch
* Each resolution "warm-starts" from the previous

### Industrial Application
NVIDIA used ProGAN to generate the first photorealistic high-resolution (1024×1024) face images, directly enabling StyleGAN's development.

---

## 11.4 BigGAN — Large Scale GAN Training (Brock et al., 2018)

### Scaling Rules

BigGAN showed that GANs benefit enormously from scale:

* **Batch size:** Increasing from 256 to 2048 dramatically improves quality
* **Channel width:** Wider networks (up to 128 channels per layer)
* **Truncation trick:** At inference, sample $$z$$ from truncated normal for higher quality at the cost of diversity

$$z \sim \text{TruncatedNormal}(0, 1, [-\psi, \psi])$$

Smaller $$\psi$$ = higher quality but lower diversity.

### Class-Conditional Batch Normalization

$$\text{CBN}(x; y) = \gamma(y) \frac{x - \mu}{\sigma} + \beta(y)$$

where $$\gamma(y)$$ and $$\beta(y)$$ are class-dependent learned parameters.

### Industrial Application
DeepMind/Google used BigGAN for ImageNet-scale generation, achieving state-of-the-art FID scores. The truncation trick is now standard in commercial image generation APIs.

---

## 11.5 Summary Comparison of GAN Variants

| Variant | Key Innovation | Best For | Limitation |
| --- | --- | --- | --- |
| Vanilla GAN | Adversarial training | Proof of concept | Mode collapse, instability |
| DCGAN | Convolutional architecture | Image generation | Limited resolution |
| CGAN | Conditional generation | Controlled output | Needs labels |
| WGAN-GP | Wasserstein distance | Stable training | Slower (5 critic steps) |
| InfoGAN | Disentangled representations | Interpretable generation | Complex training |
| Pix2Pix | Paired translation | Domain conversion | Needs paired data |
| CycleGAN | Unpaired translation | Style transfer | Geometry changes hard |
| ProGAN | Progressive growing | High-resolution | Complex scheduling |
| StyleGAN | Style-based synthesis | Faces, controlled gen | Compute-intensive |
| BigGAN | Scale | Diverse categories | Massive compute needed |

# Chapter 12: Training Challenges and Solutions

---

## 12.1 Mode Collapse

### Definition
The generator learns to produce only a small subset of the data distribution modes, ignoring diversity:

$$p_g \text{ concentrates on few modes while } p_{data} \text{ is multi-modal}$$

### Manifestation
* Generator produces nearly identical outputs regardless of input $$z$$
* Low variety in generated samples

### Solutions

| Solution | Mechanism |
| --- | --- |
| Minibatch discrimination | D sees batches, not single samples |
| Unrolled GANs | G optimizes against future D states |
| WGAN | Wasserstein loss provides continuous gradients |
| Mode regularization | Penalize generator for mapping different z to same output |
| Spectral normalization | Stabilize D's Lipschitz constant |

## 12.2 Training Instability

### The Oscillation Problem

G and D chase each other without converging:

$$\theta_G^{t+1} = \theta_G^t - \alpha \nabla_{\theta_G} \mathcal{L}_G(\theta_G^t, \theta_D^t)$$
$$\theta_D^{t+1} = \theta_D^t - \alpha \nabla_{\theta_D} \mathcal{L}_D(\theta_G^t, \theta_D^t)$$

This is a **non-cooperative game** — unlike single-objective optimization, there's no guarantee of convergence with gradient descent.

### Solutions

| Technique | Description |
| --- | --- |
| Two-timescale learning rates | D learns faster than G |
| Label smoothing | Use 0.9 instead of 1.0 for real labels |
| Feature matching | G matches statistics of D's intermediate layers |
| Historical averaging | Penalize deviation from running average of parameters |

## 12.3 Vanishing Gradients

### The Problem
When D is too confident:

$$D(G(z)) \approx 0 \implies \nabla_G \log(1 - D(G(z))) \approx 0$$

The generator receives no useful gradient signal.

### Solutions

| Solution | How it helps |
| --- | --- |
| Non-saturating loss | $$-\log D(G(z))$$ has stronger gradients |
| WGAN | Wasserstein distance never saturates |
| Relativistic GAN | D estimates probability that real is "more real" than fake |
| Least Squares GAN | Uses MSE loss instead of BCE |

## 12.4 Evaluation Metrics

Evaluating GANs is notoriously difficult. Key metrics:

### Inception Score (IS)
$$IS = \exp(\mathbb{E}_x [KL(p(y|x) \| p(y))])$$

Measures both quality (low entropy $$p(y|x)$$) and diversity (high entropy $$p(y)$$).

### Fréchet Inception Distance (FID)
$$FID = \|\mu_r - \mu_g\|^2 + \text{Tr}(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2})$$

Compares the statistics of generated and real image features from an Inception network. **Lower is better.**

### Kernel Inception Distance (KID)
Similar to FID but uses an unbiased estimator based on the polynomial kernel. Better for small sample sizes.

In [0]:
# ============================================================
# TRAINING STABILIZATION TECHNIQUES - Demonstrations
# ============================================================

# ----- Technique 1: Spectral Normalization -----
class SpectralNormDiscriminator(nn.Module):
    """
    Discriminator with Spectral Normalization (Miyato et al., 2018).
    Controls the Lipschitz constant of D by normalizing weight matrices
    by their spectral norm (largest singular value).
    
    This ensures: ||D(x1) - D(x2)|| <= ||x1 - x2|| (1-Lipschitz)
    """
    def __init__(self, img_dim=784):
        super(SpectralNormDiscriminator, self).__init__()
        self.model = nn.Sequential(
            nn.utils.spectral_norm(nn.Linear(img_dim, 512)),
            nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Linear(512, 256)),
            nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Linear(256, 1)),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x)


# ----- Technique 2: Label Smoothing -----
def smooth_labels(batch_size, device, real_range=(0.8, 1.0), fake_range=(0.0, 0.2)):
    """
    One-sided label smoothing: prevents D from becoming overconfident.
    Instead of 1.0 for real, use values in [0.8, 1.0]
    Instead of 0.0 for fake, use values in [0.0, 0.2]
    """
    real_labels = torch.FloatTensor(batch_size, 1).uniform_(*real_range).to(device)
    fake_labels = torch.FloatTensor(batch_size, 1).uniform_(*fake_range).to(device)
    return real_labels, fake_labels


# ----- Technique 3: Feature Matching -----
class FeatureMatchingLoss(nn.Module):
    """
    Instead of maximizing D's output, G matches intermediate features.
    Loss = ||E[f(x_real)] - E[f(G(z))]||^2
    where f(x) are intermediate layer activations of D.
    """
    def __init__(self):
        super(FeatureMatchingLoss, self).__init__()
    
    def forward(self, real_features, fake_features):
        loss = 0
        for rf, ff in zip(real_features, fake_features):
            loss += torch.mean((rf.mean(0) - ff.mean(0)) ** 2)
        return loss


# ----- Technique 4: Minibatch Discrimination -----
class MinibatchDiscrimination(nn.Module):
    """
    Allows D to look at multiple samples in a batch simultaneously.
    Helps detect mode collapse since all generated samples would
    have similar features, making them easy to distinguish.
    """
    def __init__(self, in_features, out_features, kernel_dim=5):
        super(MinibatchDiscrimination, self).__init__()
        self.T = nn.Parameter(torch.randn(in_features, out_features, kernel_dim))
    
    def forward(self, x):
        # x shape: (batch_size, in_features)
        # Compute pairwise distances between samples in the batch
        matrices = torch.mm(x, self.T.view(x.size(1), -1))
        matrices = matrices.view(x.size(0), -1, self.T.size(2))
        
        # L1 distance between all pairs
        diffs = matrices.unsqueeze(0) - matrices.unsqueeze(1)
        abs_diffs = torch.abs(diffs).sum(2)
        minibatch_features = torch.exp(-abs_diffs).sum(0)
        
        return torch.cat([x, minibatch_features], dim=1)


# Demonstrate label smoothing
real_smooth, fake_smooth = smooth_labels(8, device)
print("Training Stabilization Techniques:")
print("=" * 50)
print(f"\n1. Spectral Normalization: Constrains D's weight matrices")
print(f"   Parameters: {sum(p.numel() for p in SpectralNormDiscriminator().parameters()):,}")
print(f"\n2. Label Smoothing Example:")
print(f"   Real labels (instead of 1.0): {real_smooth[:4].squeeze().tolist()}")
print(f"   Fake labels (instead of 0.0): {fake_smooth[:4].squeeze().tolist()}")
print(f"\n3. Feature Matching: Minimizes distance in feature space")
print(f"\n4. Minibatch Discrimination: D sees batch-level statistics")

# Chapter 13: Advanced Topics and Modern Extensions

---

## 13.1 Least Squares GAN (LSGAN)

Replaces BCE loss with MSE, providing non-vanishing gradients far from the decision boundary:

$$\mathcal{L}_D = \frac{1}{2}\mathbb{E}_{x \sim p_{data}}[(D(x) - 1)^2] + \frac{1}{2}\mathbb{E}_{z \sim p_z}[(D(G(z)))^2]$$

$$\mathcal{L}_G = \frac{1}{2}\mathbb{E}_{z \sim p_z}[(D(G(z)) - 1)^2]$$

**Advantage:** Penalizes samples that are correctly classified but far from the decision boundary, pushing them toward the boundary and thus toward the real data manifold.

## 13.2 Relativistic GAN (RaGAN)

The standard discriminator estimates $$P(\text{real})$$ independently for each sample. The relativistic discriminator estimates the probability that a real sample is **more realistic than a fake sample**:

$$D(x_{real}, x_{fake}) = \sigma(C(x_{real}) - C(x_{fake}))$$

where $$C$$ is the critic function (no sigmoid).

This makes both real and fake data contribute to both G and D updates, improving training stability.

## 13.3 Self-Attention GAN (SAGAN)

Convolutions operate locally. SAGAN (Zhang et al., 2018) adds **self-attention** to capture long-range dependencies:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

where:
* $$Q = W_q \cdot f(x)$$ (query)
* $$K = W_k \cdot f(x)$$ (key)  
* $$V = W_v \cdot f(x)$$ (value)

This allows the generator to coordinate details across spatially distant regions (e.g., ensuring both eyes of a face look in the same direction).

## 13.4 GANs for Text (SeqGAN, TextGAN)

Applying GANs to discrete text is challenging because:
* Text is discrete (can't backpropagate through argmax/sampling)
* No meaningful gradient from D to G for discrete outputs

**SeqGAN** (Yu et al., 2017) solves this using **reinforcement learning**:
* Generator = policy network
* Discriminator = reward function
* Training via REINFORCE / policy gradient

$$\nabla_{\theta} J(\theta) = \sum_{t=1}^{T} \mathbb{E}_{y_t \sim G_\theta}[\nabla_\theta \log G_\theta(y_t | y_{1:t-1}) \cdot Q_{D_\phi}(y_{1:t-1}, y_t)]$$

## 13.5 GANs for Tabular Data (CTGAN, TableGAN)

**CTGAN** (Xu et al., 2019) adapts GANs for mixed-type tabular data:
* Uses **mode-specific normalization** for continuous columns
* Uses **conditional training** to handle class imbalance
* Employs **PacGAN** (packing multiple samples) to prevent mode collapse

### Industrial Application
JPMorgan Chase, Capital One, and other banks use CTGAN to generate synthetic customer datasets for model development, allowing data scientists to work without accessing real PII data.

## 13.6 Diffusion Models vs. GANs

Since 2020, **diffusion models** (DDPM, Stable Diffusion, DALL-E) have surpassed GANs in many image generation benchmarks:

| Aspect | GANs | Diffusion Models |
| --- | --- | --- |
| Training stability | Unstable (minimax game) | Stable (single objective) |
| Mode coverage | Prone to mode collapse | Full distribution coverage |
| Generation speed | Fast (single forward pass) | Slow (many denoising steps) |
| Sample quality | High (but diverse is hard) | Very high |
| Controllability | Via conditioning | Via conditioning + guidance |

However, GANs remain preferred for **real-time applications** (video synthesis, interactive tools) due to their single-pass generation speed.

In [0]:
# ============================================================
# LSGAN - Least Squares GAN Implementation
# ============================================================

def train_lsgan(num_epochs=10, latent_dim=100, lr=0.0002):
    """
    Least Squares GAN training.
    
    Key difference: Uses MSE loss instead of BCE.
    L_D = 0.5 * E[(D(x)-1)^2] + 0.5 * E[D(G(z))^2]
    L_G = 0.5 * E[(D(G(z))-1)^2]
    
    Benefits:
    - More stable gradients (quadratic penalty)
    - Penalizes samples far from decision boundary
    - Generates higher quality samples empirically
    """
    # Reuse VanillaGenerator but discriminator has NO sigmoid
    class LSGANDiscriminator(nn.Module):
        def __init__(self, img_dim=784):
            super(LSGANDiscriminator, self).__init__()
            self.model = nn.Sequential(
                nn.Linear(img_dim, 1024),
                nn.LeakyReLU(0.2),
                nn.Dropout(0.3),
                nn.Linear(1024, 512),
                nn.LeakyReLU(0.2),
                nn.Dropout(0.3),
                nn.Linear(512, 256),
                nn.LeakyReLU(0.2),
                nn.Linear(256, 1)  # NO sigmoid - raw output for MSE
            )
        def forward(self, x):
            return self.model(x)
    
    G = VanillaGenerator(latent_dim=latent_dim).to(device)
    D = LSGANDiscriminator().to(device)
    
    optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    
    # MSE Loss instead of BCE
    mse_loss = nn.MSELoss()
    
    g_losses, d_losses = [], []
    fixed_noise = torch.randn(16, latent_dim, device=device)
    
    print(f"Training LSGAN for {num_epochs} epochs...")
    print("Loss function: Mean Squared Error (not Binary Cross-Entropy)")
    print("=" * 60)
    
    for epoch in range(num_epochs):
        for batch_idx, (real_images, _) in enumerate(dataloader):
            batch_size_curr = real_images.size(0)
            real_images = real_images.view(batch_size_curr, -1).to(device)
            
            real_labels = torch.ones(batch_size_curr, 1, device=device)
            fake_labels = torch.zeros(batch_size_curr, 1, device=device)
            
            # === Train Discriminator with MSE ===
            optimizer_D.zero_grad()
            
            outputs_real = D(real_images)
            d_loss_real = mse_loss(outputs_real, real_labels)  # (D(x) - 1)^2
            
            z = torch.randn(batch_size_curr, latent_dim, device=device)
            fake_images = G(z)
            outputs_fake = D(fake_images.detach())
            d_loss_fake = mse_loss(outputs_fake, fake_labels)  # D(G(z))^2
            
            d_loss = 0.5 * (d_loss_real + d_loss_fake)
            d_loss.backward()
            optimizer_D.step()
            
            # === Train Generator with MSE ===
            optimizer_G.zero_grad()
            
            z = torch.randn(batch_size_curr, latent_dim, device=device)
            fake_images = G(z)
            outputs = D(fake_images)
            g_loss = 0.5 * mse_loss(outputs, real_labels)  # (D(G(z)) - 1)^2
            g_loss.backward()
            optimizer_G.step()
            
            g_losses.append(g_loss.item())
            d_losses.append(d_loss.item())
        
        print(f"Epoch [{epoch+1}/{num_epochs}] | "
              f"D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")
    
    print("\nTraining Complete!")
    plot_losses(g_losses, d_losses, "LSGAN Training Losses")
    
    G.eval()
    with torch.no_grad():
        generated = G(fixed_noise).view(-1, 1, 28, 28)
    show_generated_images(generated, title="LSGAN - Generated MNIST Digits")
    
    return G, D

# Train LSGAN
lsgan_G, lsgan_D = train_lsgan(num_epochs=10)

# Chapter 14: Comprehensive Industrial Applications

---

## 14.1 Healthcare & Life Sciences

| Application | GAN Variant | Company/Institution | Details |
| --- | --- | --- | --- |
| Medical image synthesis | DCGAN, ProGAN | NVIDIA (Clara) | Generate CT/MRI scans for rare conditions |
| Drug discovery | WGAN | Insilico Medicine | Generate novel molecular structures |
| Protein structure | WGAN-GP | DeepMind | Generate protein conformations |
| Data augmentation for rare diseases | CGAN | Stanford Medicine | Augment training data for rare pathology detection |
| De-identification | CycleGAN | Multiple hospitals | Remove patient identity while preserving pathology |

## 14.2 Finance & Banking

| Application | GAN Variant | Company | Details |
| --- | --- | --- | --- |
| Synthetic transaction data | CTGAN | JPMorgan, Mastercard | Privacy-preserving model development |
| Fraud detection augmentation | CGAN | American Express | Generate rare fraud patterns |
| Market simulation | TimeGAN | Two Sigma, DE Shaw | Generate realistic market scenarios |
| Stress testing | WGAN | Central banks | Generate extreme market conditions |

## 14.3 Autonomous Vehicles

| Application | GAN Variant | Company | Details |
| --- | --- | --- | --- |
| Sim-to-real transfer | CycleGAN | Waymo, Cruise | Make simulated data photorealistic |
| Weather augmentation | CGAN | Tesla, Mobileye | Generate rain/snow/fog conditions |
| Rare scenario generation | CGAN | Uber ATG | Generate edge cases (accidents, unusual objects) |
| LiDAR data augmentation | PointGAN | Velodyne, Waymo | Generate synthetic 3D point clouds |

## 14.4 Creative Industries

| Application | GAN Variant | Company | Details |
| --- | --- | --- | --- |
| Face generation | StyleGAN | NVIDIA, Generated Photos | Marketing, gaming avatars |
| Art creation | CycleGAN, StyleGAN | Obvious (Portrait of Edmond de Belamy) | First AI art sold at Christie's ($432K) |
| Music generation | WaveGAN | OpenAI, Google Magenta | Generate audio waveforms |
| Video synthesis | Vid2Vid | NVIDIA | Video-to-video translation |
| Fashion design | CGAN | Zalando, Stitch Fix | Generate clothing designs from sketches |

## 14.5 Manufacturing & Engineering

| Application | GAN Variant | Company | Details |
| --- | --- | --- | --- |
| Defect detection training | CGAN | Siemens, Bosch | Generate defective part images |
| Material design | WGAN | Citrine Informatics | Generate novel material compositions |
| Topology optimization | CGAN | Autodesk | Generate structural designs |
| Chip design | DCGAN | Synopsys | Generate circuit layout variations |

## 14.6 Retail & E-Commerce

| Application | GAN Variant | Company | Details |
| --- | --- | --- | --- |
| Virtual try-on | CycleGAN | Zalando, Amazon | Try clothes on virtual avatars |
| Product image generation | StyleGAN | Shopify, Wayfair | Generate product photos without photoshoots |
| Recommendation diversity | GAN-based | Netflix | Diversify thumbnail recommendations |
| Synthetic reviews (detection) | SeqGAN | Amazon, Yelp | Generate fake reviews to train detectors |

In [0]:
# ============================================================
# TABULAR GAN - Generating Synthetic Structured Data
# ============================================================
# Industrial Application: Privacy-preserving synthetic data generation
# for financial services, healthcare, and telecommunications.

class TabularGenerator(nn.Module):
    """
    Generator for tabular (structured) data.
    Handles mixed data types: continuous + categorical features.
    
    Industrial Use: Generate synthetic customer profiles, transaction
    records, or patient data for model development without PII exposure.
    """
    def __init__(self, latent_dim=32, output_dim=10):
        super(TabularGenerator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            
            nn.Linear(64, output_dim),
            nn.Tanh()  # Normalized output
        )
    
    def forward(self, z):
        return self.model(z)


class TabularDiscriminator(nn.Module):
    """
    Discriminator for tabular data with PacGAN-style input.
    Uses dropout heavily to prevent overfitting on small datasets.
    """
    def __init__(self, input_dim=10):
        super(TabularDiscriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.5),
            
            nn.Linear(64, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.5),
            
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.5),
            
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x)


# Generate synthetic tabular data (simulating customer features)
def generate_synthetic_customers(n_real=1000, n_features=10):
    """
    Simulate a use case: generating synthetic customer data.
    Real data has correlations (e.g., age correlates with income).
    """
    # Create correlated synthetic "real" data
    np.random.seed(42)
    age = np.random.normal(40, 12, n_real)
    income = age * 1000 + np.random.normal(0, 5000, n_real)
    credit_score = np.clip(300 + age * 8 + np.random.normal(0, 50, n_real), 300, 850)
    balance = np.abs(income * 0.3 + np.random.normal(0, 2000, n_real))
    
    # Normalize to [-1, 1]
    features = np.column_stack([age, income, credit_score, balance])
    features_normalized = 2 * (features - features.min(0)) / (features.max(0) - features.min(0)) - 1
    
    # Pad to n_features dimensions with noise
    if n_features > 4:
        extra = np.random.normal(0, 0.5, (n_real, n_features - 4))
        features_normalized = np.column_stack([features_normalized, extra])
    
    return torch.FloatTensor(features_normalized), features


# Train Tabular GAN
def train_tabular_gan(num_epochs=200, latent_dim=32, n_features=10):
    real_data, original_data = generate_synthetic_customers(n_real=1000, n_features=n_features)
    real_data = real_data.to(device)
    
    G = TabularGenerator(latent_dim=latent_dim, output_dim=n_features).to(device)
    D = TabularDiscriminator(input_dim=n_features).to(device)
    
    optimizer_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))
    criterion = nn.BCELoss()
    
    print("Training Tabular GAN for synthetic customer data generation...")
    print(f"Real data shape: {real_data.shape}")
    print("=" * 50)
    
    for epoch in range(num_epochs):
        # Train D
        optimizer_D.zero_grad()
        real_labels = torch.ones(real_data.size(0), 1, device=device)
        fake_labels = torch.zeros(real_data.size(0), 1, device=device)
        
        output_real = D(real_data)
        d_loss_real = criterion(output_real, real_labels)
        
        z = torch.randn(real_data.size(0), latent_dim, device=device)
        fake_data = G(z)
        output_fake = D(fake_data.detach())
        d_loss_fake = criterion(output_fake, fake_labels)
        
        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        optimizer_D.step()
        
        # Train G
        optimizer_G.zero_grad()
        z = torch.randn(real_data.size(0), latent_dim, device=device)
        fake_data = G(z)
        output = D(fake_data)
        g_loss = criterion(output, real_labels)
        g_loss.backward()
        optimizer_G.step()
        
        if (epoch + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] | D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")
    
    # Generate synthetic data and compare distributions
    G.eval()
    with torch.no_grad():
        z = torch.randn(1000, latent_dim, device=device)
        synthetic_data = G(z).cpu().numpy()
    
    # Compare first 4 features (meaningful ones)
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    feature_names = ['Age', 'Income', 'Credit Score', 'Balance']
    
    for i, (ax, name) in enumerate(zip(axes.flat, feature_names)):
        ax.hist(real_data[:, i].cpu().numpy(), bins=30, alpha=0.5, label='Real', density=True)
        ax.hist(synthetic_data[:, i], bins=30, alpha=0.5, label='Synthetic', density=True)
        ax.set_title(f'{name} Distribution: Real vs Synthetic')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Tabular GAN: Synthetic Customer Data Generation\n(Industrial Use: Privacy-Preserving Analytics)', fontsize=13)
    plt.tight_layout()
    plt.show()
    
    return G

tab_G = train_tabular_gan(num_epochs=200)

# Chapter 15: GAN Evaluation and Model Comparison

---

## 15.1 Evaluation Metrics Deep Dive

### Fréchet Inception Distance (FID) — The Gold Standard

FID models both real and generated distributions as multivariate Gaussians in Inception-v3 feature space:

$$FID = \|\mu_r - \mu_g\|^2 + \text{Tr}\left(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}\right)$$

where:
* $$\mu_r, \Sigma_r$$ are the mean and covariance of real image features
* $$\mu_g, \Sigma_g$$ are the mean and covariance of generated image features

**Lower FID = Better quality and diversity.**

### Precision and Recall for GANs

* **Precision**: Fraction of generated samples that fall within the real data manifold (quality)
* **Recall**: Fraction of the real data manifold covered by generated samples (diversity)

### Perceptual Path Length (PPL)

Measures smoothness of the latent space:

$$PPL = \mathbb{E}\left[\frac{1}{\epsilon^2} d\left(G(\text{lerp}(z_1, z_2, t)), G(\text{lerp}(z_1, z_2, t+\epsilon))\right)\right]$$

Lower PPL indicates a more disentangled, smoother latent space.

## 15.2 Practical Guidelines for Choosing a GAN

| Scenario | Recommended GAN | Reason |
| --- | --- | --- |
| Quick prototyping | Vanilla GAN | Simple, fast to implement |
| Image generation (general) | DCGAN + WGAN-GP | Stable, good quality |
| Controlled generation (with labels) | CGAN | Specify desired output class |
| High-resolution faces | StyleGAN2/3 | State-of-the-art quality |
| Domain transfer (paired data) | Pix2Pix | Best with paired examples |
| Domain transfer (unpaired) | CycleGAN | No paired data needed |
| Tabular/structured data | CTGAN / TableGAN | Handles mixed types |
| Time series | TimeGAN | Preserves temporal dynamics |
| Very stable training required | WGAN-GP | Wasserstein + gradient penalty |
| Large-scale diverse generation | BigGAN | Scales with compute |

In [0]:
# ============================================================
# GAN EVALUATION - Comparing Generated Samples
# ============================================================

def compute_simple_fid(real_features, fake_features):
    """
    Simplified FID computation using raw pixel features.
    (Production FID uses Inception-v3 features, but this demonstrates the concept.)
    
    FID = ||mu_r - mu_g||^2 + Tr(Sigma_r + Sigma_g - 2*sqrt(Sigma_r * Sigma_g))
    """
    # Compute statistics
    mu_real = np.mean(real_features, axis=0)
    mu_fake = np.mean(fake_features, axis=0)
    sigma_real = np.cov(real_features, rowvar=False)
    sigma_fake = np.cov(fake_features, rowvar=False)
    
    # Mean difference
    diff = mu_real - mu_fake
    mean_diff_sq = np.dot(diff, diff)
    
    # Matrix square root (simplified - using eigendecomposition)
    # For proper FID, use scipy.linalg.sqrtm
    product = sigma_real @ sigma_fake
    eigenvalues = np.linalg.eigvalsh(product)
    eigenvalues = np.maximum(eigenvalues, 0)  # Numerical stability
    sqrt_product_trace = np.sum(np.sqrt(eigenvalues))
    
    trace_term = np.trace(sigma_real) + np.trace(sigma_fake) - 2 * sqrt_product_trace
    
    fid = mean_diff_sq + trace_term
    return fid


def evaluate_gan_quality(generator, latent_dim, is_conv=False, name="GAN"):
    """
    Evaluate a trained GAN by computing:
    1. Visual quality (display samples)
    2. Diversity (variance of generated samples)
    3. Simplified FID score
    """
    generator.eval()
    
    # Generate samples
    n_samples = 500
    with torch.no_grad():
        if is_conv:
            z = torch.randn(n_samples, latent_dim, 1, 1, device=device)
            generated = generator(z).cpu().numpy().reshape(n_samples, -1)
        else:
            z = torch.randn(n_samples, latent_dim, device=device)
            generated = generator(z).cpu().numpy()
    
    # Real samples
    real_samples = []
    for imgs, _ in dataloader:
        real_samples.append(imgs.numpy().reshape(imgs.size(0), -1))
        if len(real_samples) * batch_size >= n_samples:
            break
    real_features = np.concatenate(real_samples)[:n_samples]
    
    # Compute metrics
    diversity = np.std(generated)
    fid = compute_simple_fid(real_features, generated)
    
    return {
        'name': name,
        'fid': fid,
        'diversity': diversity,
        'mean_pixel': np.mean(generated),
        'std_pixel': np.std(generated)
    }


# Compare all trained models
print("\n" + "=" * 60)
print("GAN MODEL COMPARISON")
print("=" * 60)

results = []

# Evaluate each trained model
if 'vanilla_G' in dir():
    results.append(evaluate_gan_quality(vanilla_G, 100, is_conv=False, name="Vanilla GAN"))
if 'dc_G_trained' in dir():
    results.append(evaluate_gan_quality(dc_G_trained, 100, is_conv=True, name="DCGAN"))
if 'wgan_G' in dir():
    results.append(evaluate_gan_quality(wgan_G, 100, is_conv=False, name="WGAN-GP"))
if 'lsgan_G' in dir():
    results.append(evaluate_gan_quality(lsgan_G, 100, is_conv=False, name="LSGAN"))

if results:
    print(f"\n{'Model':<15} {'FID (lower=better)':<22} {'Diversity':<12} {'Mean Pixel':<12}")
    print("-" * 65)
    for r in results:
        print(f"{r['name']:<15} {r['fid']:<22.2f} {r['diversity']:<12.4f} {r['mean_pixel']:<12.4f}")
    
    # Visualization comparison
    fig, axes = plt.subplots(1, len(results), figsize=(5*len(results), 5))
    if len(results) == 1:
        axes = [axes]
    
    for ax, r in zip(axes, results):
        ax.bar(['FID'], [r['fid']], color='coral', alpha=0.7)
        ax.set_title(f"{r['name']}\nFID: {r['fid']:.1f}")
        ax.set_ylabel('FID Score (lower is better)')
    
    plt.suptitle('GAN Variant Comparison - FID Scores', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No trained models available. Run the training cells above first.")

# Chapter 16: Summary and Key Takeaways

---

## 16.1 Core Principles Recap

1. **Adversarial Training**: Two networks competing drives both to improve — the fundamental insight behind GANs

2. **The Minimax Game**:
$$\min_G \max_D \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

3. **Convergence**: At equilibrium, $$p_g = p_{data}$$ and $$D(x) = \frac{1}{2}$$ everywhere

4. **No explicit density estimation**: GANs learn to sample from $$p_{data}$$ without modeling it explicitly

## 16.2 Evolution Summary

```
2014: Vanilla GAN (proof of concept)
  │
  ├── 2014: CGAN (add conditions)
  │
  ├── 2015: DCGAN (convolutional architecture)
  │     │
  │     ├── 2016: Pix2Pix (paired translation)
  │     │
  │     └── 2017: CycleGAN (unpaired translation)
  │
  ├── 2017: WGAN/WGAN-GP (training stability)
  │
  ├── 2017: ProGAN (progressive growing)
  │     │
  │     └── 2018: StyleGAN (style control)
  │           │
  │           ├── 2020: StyleGAN2
  │           └── 2021: StyleGAN3
  │
  └── 2018: BigGAN (scale)
       │
       └── 2019: CTGAN (tabular data)
```

## 16.3 When to Use GANs vs. Alternatives

| Use Case | GAN | VAE | Diffusion | Flow |
| --- | --- | --- | --- | --- |
| Real-time generation | Best | Good | Slow | Good |
| Sample quality | Excellent | Good | Best | Good |
| Training stability | Difficult | Easy | Easy | Easy |
| Mode coverage | Poor-Medium | Good | Best | Good |
| Latent space quality | Medium | Best | N/A | Best |
| Likelihood estimation | No | Yes | Yes | Yes |

## 16.4 The Future of GANs

While diffusion models have overtaken GANs for static image generation, GANs remain relevant for:
* **Real-time applications** (video games, AR/VR, interactive tools)
* **Domain adaptation** (sim-to-real, cross-modality)
* **Data augmentation** (medical, financial, autonomous driving)
* **Super-resolution** (upscaling, enhancement)
* **Hybrid architectures** (GAN + Diffusion combinations)

---

## 16.5 Key References

1. Goodfellow, I. et al. (2014). "Generative Adversarial Nets." NeurIPS.
2. Radford, A. et al. (2015). "Unsupervised Representation Learning with DCGANs." ICLR.
3. Mirza, M. & Osindero, S. (2014). "Conditional Generative Adversarial Nets." arXiv.
4. Arjovsky, M. et al. (2017). "Wasserstein GAN." ICML.
5. Gulrajani, I. et al. (2017). "Improved Training of WGANs." NeurIPS.
6. Karras, T. et al. (2017). "Progressive Growing of GANs." ICLR.
7. Karras, T. et al. (2019). "A Style-Based Generator Architecture for GANs." CVPR.
8. Zhu, J.-Y. et al. (2017). "Unpaired Image-to-Image Translation using Cycle-Consistent Adversarial Networks." ICCV.
9. Brock, A. et al. (2018). "Large Scale GAN Training for High Fidelity Natural Image Synthesis." ICLR.
10. Xu, L. et al. (2019). "Modeling Tabular Data using Conditional GAN." NeurIPS.

---

*This notebook provides a comprehensive foundation for understanding GANs. For production implementations, consider using established libraries like PyTorch-GAN, StyleGAN2-ADA (NVIDIA), or CTGAN (DataCebo) which implement additional optimizations, mixed-precision training, and multi-GPU support.*